## Ch8-05 -- verification def closes the anatomy

This notebook introduces `verification def EnergyConservationTest`; after running it you can declare a named verification case whose objective verifies `energyConservationReq`.

[Ch8-04](04-energy-conservation-req.ipynb) added `EnergyConservationReq` and its usage `energyConservationReq`, and showed `deliveredEnergyBoundedBySupply` is now tied to it. A complete requirement has three parts (Part 4): description, rationale, and verification method. The first two are in `EnergyConservationReq`'s `doc` comment. `verification def` in SysML v2 (SS7.24) closes the anatomy by declaring how the requirement will be verified: the subject under test, and an objective that names the requirement usage to verify -- the same construct [Ch3-04](../ch03-measures/04-verification-case.ipynb) introduced for `TimelyToastTest`, mirrored here exactly.

In [1]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")

# editor.add_verification_def(owner='ToasterDemo', name='EnergyConservationTest', doc=..., subject_type='HeatGenerator') when API ships
# spec: SysML v2 formal/2026-03-02 SS7.24.2 (VerificationCaseDefinition)
VERIF_DEF_OPEN = "verification def EnergyConservationTest {"
print(VERIF_DEF_OPEN)

verification def EnergyConservationTest {


`verification def EnergyConservationTest` declares a reusable verification case for the energy conservation requirement. Like `requirement def`, a `verification def` takes a subject (the system under test) and a body specifying what must be determined.

In [2]:
# editor.set_doc(owner='ToasterDemo::EnergyConservationTest', text=...) when API ships
# spec: SysML v2 formal/2026-03-02 SS7.21.2 (informal text applies to all elements including verification defs)
# Note: #verificationMethod = VerificationMethodKind::analysis metadata not yet supported (toaster#19 / OpenSysML#608)
# spec: SysML v2 formal/2026-03-02 SS7.24 Table 22 (Verification Methods Compartment)
DOC_COMMENT = """\
    doc /*
     * Verification method: analysis (the Z3-backed model-checked proof
     * Ch8-02's verify_holds() already runs over
     * deliveredEnergyBoundedBySupply, which energyConservationReq's own
     * require constraint subsets).
     * Method type: analysis (VerificationMethodKind::analysis, SysML v2
     * formal/2026-03-02 SS7.24 Table 22).
     * Note: formal #verificationMethod metadata not yet supported in
     * OpenSysML v0.9.0; tracked at toaster#19 / OpenSysML#608.
     * spec: SysML v2 formal/2026-03-02 SS7.24.2 (VerificationCaseDefinition).
     */"""
print(DOC_COMMENT)

    doc /*
     * Verification method: analysis (the Z3-backed model-checked proof
     * Ch8-02's verify_holds() already runs over
     * deliveredEnergyBoundedBySupply, which energyConservationReq's own
     * require constraint subsets).
     * Method type: analysis (VerificationMethodKind::analysis, SysML v2
     * formal/2026-03-02 SS7.24 Table 22).
     * Note: formal #verificationMethod metadata not yet supported in
     * OpenSysML v0.9.0; tracked at toaster#19 / OpenSysML#608.
     * spec: SysML v2 formal/2026-03-02 SS7.24.2 (VerificationCaseDefinition).
     */


The `doc` block holds the informal text of the verification case. It describes the verification method as analysis: Ch8-02's `verify_holds()` already runs the Z3-backed proof this verification case points at, over a lemma `energyConservationReq`'s own constraint now subsets. The formal `#verificationMethod = VerificationMethodKind::analysis` metadata annotation (SS7.24 Table 22) is the spec-defined way to declare the method kind; it is not yet supported in OpenSysML v0.9.0 (toaster#19 / OpenSysML#608).

In [3]:
# editor.set_subject(owner='ToasterDemo::EnergyConservationTest', subject_type='HeatGenerator') when API ships
SUBJECT_DECL = "    subject heatGen : HeatGenerator;"
print(SUBJECT_DECL)

    subject heatGen : HeatGenerator;


The `subject heatGen : HeatGenerator` declaration names the entity under test: any `HeatGenerator` instance placed in this verification role.

In [4]:
# editor.set_objective(owner='ToasterDemo::EnergyConservationTest', verify=['energyConservationReq']) when API ships
# spec: SysML v2 formal/2026-03-02 SS7.24.2: objective declares what requirements are verified
OBJECTIVE_BODY = """    objective {
        verify energyConservationReq;
    }"""
print(OBJECTIVE_BODY)

    objective {
        verify energyConservationReq;
    }


The `objective` block names the requirement usage being verified: `energyConservationReq : EnergyConservationReq` from [Ch8-04](04-energy-conservation-req.ipynb). `verify energyConservationReq` is a shorthand satisfy constraint (SS7.24.2) that asserts the objective is met when `energyConservationReq` is determined to hold for the subject. `verify` takes a requirement **usage**, not a definition: `verify EnergyConservationReq` would fail because `EnergyConservationReq` is a `requirementDef`, not a usage -- the same rule [Ch3-04](../ch03-measures/04-verification-case.ipynb) confirmed for `TimelyToastTest`.

In [5]:
TOASTER_INCREMENT = f"{VERIF_DEF_OPEN}\n{DOC_COMMENT}\n{SUBJECT_DECL}\n{OBJECTIVE_BODY}\n}}"
print(TOASTER_INCREMENT)

source = Path("../../models/ch08-cumulative.sysml").read_text()
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

verification def EnergyConservationTest {
    doc /*
     * Verification method: analysis (the Z3-backed model-checked proof
     * Ch8-02's verify_holds() already runs over
     * deliveredEnergyBoundedBySupply, which energyConservationReq's own
     * require constraint subsets).
     * Method type: analysis (VerificationMethodKind::analysis, SysML v2
     * formal/2026-03-02 SS7.24 Table 22).
     * Note: formal #verificationMethod metadata not yet supported in
     * OpenSysML v0.9.0; tracked at toaster#19 / OpenSysML#608.
     * spec: SysML v2 formal/2026-03-02 SS7.24.2 (VerificationCaseDefinition).
     */
    subject heatGen : HeatGenerator;
    objective {
        verify energyConservationReq;
    }
}


The verification case loads without diagnostics. The next cell checks what happens when its objective names a requirement definition instead of a usage.

In [6]:
# Negative control: verify referencing a requirement def (not a usage) raises a type error.
bad_source = """
package Bad {
    private import ScalarValues::*;
    part def HeatGen { attribute power : Real default = 800.0; }
    requirement def EnergyConservationReq {
        subject heatGen : HeatGen;
        require constraint { heatGen.power >= 0.0 }
    }
    verification def BadCheck {
        subject heatGen : HeatGen;
        objective {
            verify EnergyConservationReq;  // error: must be a requirement usage, not a def
        }
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected parse/semantic error for verify-on-def"
print("Expected error:", bad.diagnostics[0].message)

Expected error: satisfy target must be a requirement usage, found requirementDef


With the negative control confirmed, the next cell looks up the verification case in the loaded model.

In [7]:
vd = model.find("ToasterDemo::EnergyConservationTest")
assert vd is not None
print(f"verification kind : {vd.kind}")
print(f"verification id   : {vd.id}")

for e in model.query():
    d = e.as_dict()
    if d["@type"] == "VerificationCaseDefinition" and d["qualifiedName"] == "ToasterDemo::EnergyConservationTest":
        print(f"VerificationCaseDefinition: {d['qualifiedName']}")
conn.close()

verification kind : verificationCaseDef
verification id   : ToasterDemo::EnergyConservationTest
VerificationCaseDefinition: ToasterDemo::EnergyConservationTest


`verification def EnergyConservationTest { doc /* ... */ subject heatGen : HeatGenerator; objective { verify energyConservationReq; } }` printed above loaded without error, and `model.find()` returns its symbol, confirmed as a `VerificationCaseDefinition` by `model.query()` above.

Try the chapter exercise in `exercises/ch08/exercise.ipynb`: it works through the same `verify_satisfaction()` and stale-detection pattern on its own, separate coffee-maker exercise model.